In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 at ang Qiskit SDK

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
```
</details>

Ang Qiskit SDK ay nagbibigay ng ilang mga kasangkapan para sa pag-convert sa pagitan ng mga OpenQASM na representasyon ng mga quantum program, at ng klase na [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Mag-import ng OpenQASM 2 program sa Qiskit
May dalawang function na nag-iimport ng mga OpenQASM 2 program sa Qiskit.
Ang mga ito ay [`qasm2.load()`](../api/qiskit/qasm2#load), na tumatanggap ng filename, at [`qasm2.loads()`](../api/qiskit/qasm2#loads), na tumatanggap ng OpenQASM 2 program bilang string.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Tingnan ang [OpenQASM 2 Qiskit API](https://docs.quantum.ibm.com/api/qiskit/qasm2) para sa karagdagang impormasyon.

### Mag-import ng simpleng mga program
Para sa karamihan ng mga OpenQASM 2 program, maaari mo lang gamitin ang `qasm2.load` at `qasm2.loads` na may isang argument.

#### Halimbawa: mag-import ng OpenQASM 2 program bilang string
Gamitin ang `qasm2.loads()` para mag-import ng OpenQASM 2 program bilang string sa isang QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Halimbawa: mag-import ng OpenQASM 2 program mula sa file
Gamitin ang `load()` para mag-import ng OpenQASM 2 program mula sa file sa isang QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### I-link ang mga OpenQASM 2 gate sa mga Qiskit gate
Sa default, ang OpenQASM 2 importer ng Qiskit ay tinatrato ang include file na `"qelib1.inc"` bilang isang *de facto* standard library.
Tinatrato ng importer ang file na ito bilang naglalaman ng eksaktong mga gate na inilalarawan nito sa [orihinal na papel na nagtatakda ng OpenQASM 2](https://arxiv.org/abs/1707.03429).
Gagamitin ng Qiskit ang mga built-in na gate sa [circuit library](../api/qiskit/circuit_library) para kumatawan sa mga gate sa `"qelib1.inc"`.
Ang mga gate na tinukoy sa program sa pamamagitan ng manu-manong OpenQASM 2 na `gate` statement ay, sa default, itatayo bilang custom na [Qiskit `Gate` subclasses](../api/qiskit/qiskit.circuit.Gate).

Maaari mong sabihin sa importer na gumamit ng mga partikular na klase ng [`Gate`](../api/qiskit/qiskit.circuit.Gate) para sa mga `gate` statement na nararamdaman nito.
Maaari mo ring gamitin ang mekanismong ito para tratuhin ang mga karagdagang pangalan ng gate bilang "built-in", ibig sabihin, hindi nangangailangan ng eksplisitong kahulugan.
Kung itutukoy mo kung aling mga klase ng gate ang gagamitin para sa mga `gate` statement na wala sa `"qelib1.inc"`, ang resultang circuit ay karaniwang magiging mas mahusay na gamitin.

> **Warning:** Simula sa Qiskit SDK v1.0, ang OpenQASM 2 *exporter* ng Qiskit (tingnan ang [I-export ang Qiskit circuit sa OpenQASM 2](#qasm2-export)) ay nagpapatuloy na kumilos na parang ang `"qelib1.inc"` ay may mas maraming gate kaysa sa talagang mayroon ito.
> Ibig sabihin nito, maaaring hindi maka-import ang default na mga setting ng importer ng isang program na na-export ng aming importer.
> Tingnan ang [ang partikular na halimbawa sa pakikipagtulungan sa legacy exporter](#qasm2-import-legacy) para malutas ang problemang ito.
> 
> Ang pagkakaibang ito ay legacy na gawi ng Qiskit, at [ito ay malulutas sa isang susunod na release ng Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

Para makapasa ng impormasyon tungkol sa custom na instruction sa OpenQASM 2 importer, gamitin ang [ang klase na `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Ito ay may apat na kinakailangang impormasyon, sa pagkakasunod:

* Ang **pangalan** ng gate, na ginagamit sa OpenQASM 2 program
* Ang **bilang ng angle parameter** na tinatanggap ng gate
* Ang **bilang ng qubit** na pinapatakbo ng gate
* Ang Python **constructor** na klase o function para sa gate, na tumatanggap ng mga parameter ng gate (ngunit hindi ang mga qubit) bilang mga indibidwal na argument

Kung makatagpo ang importer ng isang `gate` na kahulugan na tumutugma sa isang ibinigay na custom na instruction, gagamitin nito ang custom na impormasyong iyon para muling buuin ang gate object.
Kung makatagpo ng `gate` statement na tumutugma sa `name` ng isang custom na instruction, ngunit hindi tumutugma sa bilang ng mga parameter at bilang ng mga qubit, ang importer ay magtataas ng [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), upang ipahiwatig ang hindi pagkatugma sa pagitan ng ibinigay na impormasyon at ng program.

Bukod pa rito, ang ikalimang argument na `builtin` ay maaaring opsyonal na itakda sa `True` para gawing awtomatikong available ang gate sa loob ng OpenQASM 2 program, kahit na hindi ito eksplisitong tinukoy.
Kung makatagpo ang importer ng eksplisitong `gate` na kahulugan para sa isang built-in na custom na instruction, tatanggapin nito ito nang tahimik.
Tulad ng dati, kung ang eksplisitong kahulugan ng parehong pangalan ay hindi tugma sa ibinigay na custom na instruction, isang [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) ang itataas.
Ito ay kapaki-pakinabang para sa compatibility sa mga mas lumang OpenQASM 2 exporter, at sa ilang iba pang mga quantum platform na tinatrato ang "basis gates" ng kanilang hardware bilang mga built-in na instruction.

Nagbibigay ang Qiskit ng data attribute para sa pakikipagtulungan sa mga OpenQASM 2 program na ginawa ng mga legacy na bersyon ng [mga kakayahan ng OpenQASM 2 export ng Qiskit](#qasm2-export).
Ito ay [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), na maaaring ibigay bilang `custom_instructions` argument sa [`qasm2.load()`](../api/qiskit/qasm2#load) at [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Halimbawa: mag-import ng program na ginawa ng legacy exporter ng Qiskit
Ang OpenQASM 2 program na ito ay gumagamit ng mga gate na wala sa orihinal na bersyon ng `"qelib1.inc"` nang hindi ito inihahayag, ngunit mga standard na gate sa library ng Qiskit.
Maaari mong gamitin ang [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) para madaling sabihin sa importer na gamitin ang parehong hanay ng mga gate na dati nang ginagamit ng OpenQASM 2 exporter ng Qiskit.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Halimbawa: gumamit ng partikular na klase ng gate kapag nag-iimport ng OpenQASM 2 program
Hindi kayang i-verify ng Qiskit, sa pangkalahatan, kung ang kahulugan sa isang OpenQASM 2 `gate` statement ay eksaktong tumutugma sa isang standard-library gate ng Qiskit.
Sa halip, pumipili ang Qiskit ng custom na gate gamit ang eksaktong kahulugang ibinigay.
Maaaring ito ay hindi gaanong mahusay kaysa sa paggamit ng isa sa mga built-in na standard gate, o isang user-defined na custom gate.
Maaari mong manu-manong tukuyin ang mga `gate` statement na may mga partikular na klase.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Halimbawa: tukuyin ang isang bagong built-in na gate sa isang OpenQASM 2 program
Kung ang argument na `builtin=True` ay itinakda, ang isang custom na gate ay hindi na kailangang magkaroon ng kasamang kahulugan.